In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Embeddings
from sentence_transformers import SentenceTransformer

# Vector store
import chromadb
from chromadb.config import Settings

print("All imports successful!")

All imports successful!


In [3]:
df = pd.read_csv('../data/processed/complaints_cleaned.csv')
print("Shape:", df.shape)
print(df['product_clean'].value_counts())
df.head(2)

Shape: (476693, 5)
product_clean
Credit Cards        188632
Savings Accounts    154383
Money Transfers      98297
Personal Loans       35381
Name: count, dtype: int64


,product_clean,narrative_clean,narrative_length,word_count,date_received
0,Credit Cards,A card was opened under my name by a fraudster...,488,91,2025-06-13
1,Savings Accounts,I made the mistake of using my wellsfargo debi...,555,109,2025-06-13


In [4]:
# Sample 5000 per product for a balanced, manageable dataset
df_sample = df.groupby('product_clean', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 5000), random_state=42)
).reset_index(drop=True)

print("Sample shape:", df_sample.shape)
print(df_sample['product_clean'].value_counts())

Sample shape: (20000, 5)
product_clean
Credit Cards        5000
Money Transfers     5000
Personal Loans      5000
Savings Accounts    5000
Name: count, dtype: int64


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # ~500 chars per chunk
    chunk_overlap=50,      # 50 char overlap to preserve context
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    text_chunks = splitter.split_text(row['narrative_clean'])
    for i, chunk in enumerate(text_chunks):
        chunks.append({
            'chunk_id': f"{_}_{i}",
            'complaint_id': _,
            'product': row['product_clean'],
            'chunk_text': chunk,
            'chunk_index': i
        })

df_chunks = pd.DataFrame(chunks)
print("Total chunks:", len(df_chunks))
print("Avg chunks per complaint:", round(len(df_chunks) / len(df_sample), 2))
df_chunks.head(3)

100%|██████████| 20000/20000 [00:02<00:00, 9882.51it/s] 


Total chunks: 59766
Avg chunks per complaint: 2.99


,chunk_id,complaint_id,product,chunk_text,chunk_index
0,0_0,0,Credit Cards,"State Farm Bank is reporting a balance of 6,00...",0
1,1_0,1,Credit Cards,I have been back and forth with APPLE GOLDMAN ...,0
2,1_1,1,Credit Cards,. HOW CAN THEY REFUND ME FOR THE SAME EXACT IT...,1


In [6]:
# This is a lightweight but powerful model — downloads ~90MB once
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")
print("Max sequence length:", model.max_seq_length)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!
Max sequence length: 256


In [7]:
print("Generating embeddings...")

embeddings = model.encode(
    df_chunks['chunk_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", embeddings.shape)

Generating embeddings...


Batches:   0%|          | 0/934 [00:00<?, ?it/s]

Embeddings shape: (59766, 384)


In [8]:
# Create persistent vector store
chroma_client = chromadb.PersistentClient(path='../vector_store/chroma_db')

# Delete collection if it exists (clean start)
try:
    chroma_client.delete_collection("complaints")
    print("Deleted existing collection")
except:
    pass

collection = chroma_client.create_collection(
    name="complaints",
    metadata={"hnsw:space": "cosine"}
)

print("Collection created!")

Collection created!


In [9]:
# ChromaDB has a limit per batch — we insert in batches of 5000
batch_size = 5000

for i in tqdm(range(0, len(df_chunks), batch_size)):
    batch = df_chunks.iloc[i:i+batch_size]
    batch_embeddings = embeddings[i:i+batch_size]
    
    collection.add(
        ids=batch['chunk_id'].tolist(),
        embeddings=batch_embeddings.tolist(),
        documents=batch['chunk_text'].tolist(),
        metadatas=batch[['product', 'complaint_id']].to_dict('records')
    )

print("Total documents in collection:", collection.count())

100%|██████████| 12/12 [01:18<00:00,  6.55s/it]

Total documents in collection: 59766


In [10]:
# Test semantic search
query = "my credit card was charged twice for the same transaction"

query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
    include=['documents', 'metadatas', 'distances']
)

print("Query:", query)
print("\n--- Top 3 Results ---\n")
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
)):
    print(f"Result {i+1} | Product: {meta['product']} | Distance: {dist:.4f}")
    print(doc[:200])
    print()

Query: my credit card was charged twice for the same transaction

--- Top 3 Results ---

Result 1 | Product: Credit Cards | Distance: 0.2816
Starts with our wedding anniversary dinner for 280.00 for dinner for 4 people. To my surprise, my credit card  " cc ''  bill showed two lines for 280.00 each, so total of 560.00. I called my cc compan

Result 2 | Product: Credit Cards | Distance: 0.3220
The problem is with a U.S. Bank Gift Card purchased at . , the named agent on the card, has stolen 12.00. A purchase for 12.00 was made on 19. Upon checking the card balance online, the 12.00 charge w

Result 3 | Product: Credit Cards | Distance: 0.3275
Credit card  Capital one We had booked tents in , . We stayed there from , to . charges were 2300.00 charges were 3200.00 I am getting charged times this amount. time on  and second time on . Charges 



In [11]:
df_chunks.to_csv('../data/processed/complaint_chunks.csv', index=False)
print("Chunks saved! Shape:", df_chunks.shape)

Chunks saved! Shape: (59766, 5)


In [12]:
print("=== Task 2 Summary ===")
print(f"Sample complaints: {len(df_sample)}")
print(f"Total chunks: {len(df_chunks)}")
print(f"Embeddings shape: {embeddings.shape}")
print(f"Docs in ChromaDB: {collection.count()}")
print(f"\nVector store saved to: ../vector_store/chroma_db")
print(f"Chunks saved to: ../data/processed/complaint_chunks.csv")

=== Task 2 Summary ===
Sample complaints: 20000
Total chunks: 59766
Embeddings shape: (59766, 384)
Docs in ChromaDB: 59766

Vector store saved to: ../vector_store/chroma_db
Chunks saved to: ../data/processed/complaint_chunks.csv
